# One-Shot Foundation Classifier: Version 0
**ITCS 6169 Assignment 3: Foundation Model**

*By Tahiyat Chy*


## Initial Setup

In [4]:
!pip install -q open_clip_torch timm

In [5]:
# Core imports
from pathlib import Path
import random
import json
import copy

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

import open_clip

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.14.1+cu130
CUDA available: True


In [6]:
# Mount Google Drive (when running in Colab)
try:
    from google.colab import drive
    drive.mount("/content/gdrive")
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")

Mounted at /content/gdrive


In [7]:
# Path set to assignment directory
PROJECT_ROOT = Path("/content/gdrive/MyDrive/ITCS_6169/assignment3")
DATA_ROOT = PROJECT_ROOT / "data"
TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT = DATA_ROOT / "test"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Training data:", TRAIN_ROOT)
print("Testing data:", TEST_ROOT)
print("Checkpoints dir:", CHECKPOINT_DIR)

Project root: /content/gdrive/MyDrive/ITCS_6169/assignment3
Training data: /content/gdrive/MyDrive/ITCS_6169/assignment3/data/train
Testing data: /content/gdrive/MyDrive/ITCS_6169/assignment3/data/test
Checkpoints dir: /content/gdrive/MyDrive/ITCS_6169/assignment3/checkpoints


In [8]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## One-Shot Split & Dataloaders

In [9]:
BATCH_SIZE = 64
NUM_CLASSES = 16
SHOTS_PER_CLASS = 1  # Strictly 1 sample per class


class IndexedImageDataset(Dataset):
    """Loads images from (path, label) pairs using a given transform."""

    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        with Image.open(path) as img:
            img = img.convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.samples)


def build_one_shot_splits(root_dir, shots_per_class=1, seed=0):
    """
    Selects strictly `shots_per_class` (default=1) image per class for support/training,
    and places remaining images into a held-out validation pool.
    """
    rng = random.Random(seed)
    base_ds = datasets.ImageFolder(root=root_dir)
    class_names = base_ds.classes

    by_class = {c_idx: [] for c_idx in range(len(class_names))}
    for path, label in base_ds.samples:
        by_class[label].append((path, label))

    support_samples = []
    val_samples = []

    for c_idx, items in by_class.items():
        if len(items) < shots_per_class:
            raise ValueError(
                f"Class {class_names[c_idx]} has {len(items)} images (< {shots_per_class})."
            )
        shuffled = items.copy()
        rng.shuffle(shuffled)
        support_samples.extend(shuffled[:shots_per_class])
        val_samples.extend(shuffled[shots_per_class:])

    # Sort support samples by class index
    support_samples.sort(key=lambda x: x[1])
    return support_samples, val_samples, class_names

## Model & Prototype

In [10]:
DEFAULT_PROMPT_TEMPLATES = [
    "a photo of a {}.",
    "a clear photograph of a {}.",
    "a close-up photo of a {}.",
    "an image of {} in natural lighting.",
]


class OneShotFoundationClassifier(nn.Module):
    """
    Combines zero-shot text embeddings and a 1-shot visual prototype cache
    with a learnable residual logit adapter.
    """

    def __init__(
        self,
        clip_model,
        tokenizer,
        class_names,
        support_features,
        support_labels,
        templates=DEFAULT_PROMPT_TEMPLATES,
        alpha=1.2,
        beta=5.0,
        logit_scale=20.0,
    ):
        super().__init__()
        self.clip_model = clip_model
        self.tokenizer = tokenizer
        self.class_names = class_names
        self.alpha = alpha
        self.beta = beta
        self.logit_scale = logit_scale

        # Freeze foundation backbone weights
        for p in self.clip_model.parameters():
            p.requires_grad = False

        # Build text prototype embeddings averaged across prompt templates: (C, D)
        text_embeds = self._build_text_embeddings(class_names, templates)
        self.register_buffer("text_features", text_embeds)

        # Store 1-shot support visual features (N_support, D) and one-hot targets (N_support, C)
        self.register_buffer("cache_keys", F.normalize(support_features.detach(), dim=-1))
        one_hot = F.one_hot(support_labels, num_classes=len(class_names)).float()
        self.register_buffer("cache_values", one_hot)

        # Learnable residual weights over the 1-shot support cache
        self.adapter_weights = nn.Parameter(torch.ones_like(self.cache_values))

    @torch.no_grad()
    def _build_text_embeddings(self, class_names, templates):
        device = next(self.clip_model.parameters()).device
        per_class_feats = []
        for raw_name in class_names:
            readable_name = raw_name.replace("_", " ").replace("-", " ")
            prompts = [t.format(readable_name) for t in templates]
            tokens = self.tokenizer(prompts).to(device)
            feats = self.clip_model.encode_text(tokens)
            feats = F.normalize(feats, dim=-1)
            mean_feat = F.normalize(feats.mean(dim=0, keepdim=True), dim=-1)
            per_class_feats.append(mean_feat)
        return torch.cat(per_class_feats, dim=0)  # (C, D)

    @torch.no_grad()
    def encode_images(self, images):
        feats = self.clip_model.encode_image(images)
        return F.normalize(feats, dim=-1)

    def logits_from_features(self, img_features):
        """
        img_features: (B, D), L2-normalized.
        Returns combined text-similarity + one-shot visual prototype logits: (B, C).
        """
        img_features = F.normalize(img_features, dim=-1)

        # 1. Text-vision similarity logits
        text_logits = self.logit_scale * (img_features @ self.text_features.T)

        # 2. Visual cache similarity via RBF-like kernel
        # affinity: (B, N_support)
        affinity = img_features @ self.cache_keys.T
        cache_weights = torch.exp(-self.beta * (1.0 - affinity))
        weighted_values = self.cache_values * self.adapter_weights
        cache_logits = cache_weights @ weighted_values

        return text_logits + self.alpha * cache_logits

    def forward(self, images):
        with torch.no_grad():
            img_features = self.encode_images(images)
        return self.logits_from_features(img_features)

## Training & Adaptation

In [11]:
MODEL_NAME = "ViT-B-16-SigLIP"
PRETRAINED_TAG = "webli"

# Load foundation model and official preprocessing transform
clip_model, _, preprocess_val = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED_TAG, device=device
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
clip_model.eval()

# Build strict 1-shot support set (16 images) and validation set
support_samples, val_samples, class_names = build_one_shot_splits(
    TRAIN_ROOT, shots_per_class=SHOTS_PER_CLASS, seed=SEED
)
assert len(support_samples) == NUM_CLASSES * SHOTS_PER_CLASS, "Violated 1-shot rule!"

support_dataset = IndexedImageDataset(support_samples, transform=preprocess_val)
val_dataset = IndexedImageDataset(val_samples, transform=preprocess_val)

support_loader = DataLoader(support_dataset, batch_size=len(support_dataset), shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"One-shot support images: {len(support_dataset)} | Held-out val images: {len(val_dataset)}")

# Extract 1-shot support visual embeddings once
with torch.no_grad():
    sup_imgs, sup_labels = next(iter(support_loader))
    sup_imgs, sup_labels = sup_imgs.to(device), sup_labels.to(device)
    sup_feats = F.normalize(clip_model.encode_image(sup_imgs), dim=-1)

model = OneShotFoundationClassifier(
    clip_model=clip_model,
    tokenizer=tokenizer,
    class_names=class_names,
    support_features=sup_feats,
    support_labels=sup_labels,
    alpha=1.2,
    beta=5.0,
    logit_scale=20.0,
).to(device)

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  813MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/20.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

One-shot support images: 16 | Held-out val images: 2384


In [12]:
def evaluate_model(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / max(total, 1)


def fit_one_shot_adapter(model, sup_feats, sup_labels, epochs=40, lr=1e-3, weight_decay=1e-2):
    """Trains only adapter_weights on the 1-shot support features."""
    optimizer = torch.optim.AdamW([model.adapter_weights], lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    best_weights = copy.deepcopy(model.adapter_weights.detach())
    best_loss = float("inf")

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model.logits_from_features(sup_feats)
        loss = criterion(logits, sup_labels)
        loss.backward()
        optimizer.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_weights = copy.deepcopy(model.adapter_weights.detach())

    with torch.no_grad():
        model.adapter_weights.copy_(best_weights)
    return best_loss


def test_time_adapt_features(model, loader, device, steps=1, lr=5e-4):
    """
    Optional Test-Time Adaptation (entropy minimization on adapter_weights)
    over unlabeled evaluation batches without updating backbone weights.
    """
    saved_weights = copy.deepcopy(model.adapter_weights.detach())
    optimizer = torch.optim.AdamW([model.adapter_weights], lr=lr)

    model.train()
    for images, _ in loader:
        images = images.to(device)
        with torch.no_grad():
            feats = model.encode_images(images)

        for _ in range(steps):
            optimizer.zero_grad()
            logits = model.logits_from_features(feats)
            probs = F.softmax(logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1).mean()
            entropy.backward()
            optimizer.step()

    acc = evaluate_model(model, loader, device)

    # Restore post-training adapter weights if you want clean non-mutated evaluation
    with torch.no_grad():
        model.adapter_weights.copy_(saved_weights)
    return acc


zero_shot_acc = evaluate_model(model, val_loader, device)
print(f"Zero-Shot (Text Only, alpha=0) Val Acc: {zero_shot_acc:.4f}")

best_sup_loss = fit_one_shot_adapter(model, sup_feats, sup_labels, epochs=40, lr=1e-3)
one_shot_acc = evaluate_model(model, val_loader, device)
print(f"1-Shot Adapted Val Acc: {one_shot_acc:.4f} (Support Loss: {best_sup_loss:.4f})")

tta_acc = test_time_adapt_features(model, val_loader, device, steps=1, lr=5e-4)
print(f"1-Shot + TTA Val Acc: {tta_acc:.4f}")

Zero-Shot (Text Only, alpha=0) Val Acc: 0.8960
1-Shot Adapted Val Acc: 0.8960 (Support Loss: 0.9041)
1-Shot + TTA Val Acc: 0.8956
